# CineMind-AI MovieLens 25M NCF candidate training

This notebook is a handoff, not an automatic training job. It benchmarks real candidate-shard batches first. Full training remains disabled until you set the explicit opt-in below and type the confirmation phrase. The output is a separately named candidate; production artifacts are never promoted or overwritten.

Before continuing, export the candidate with `training/colab_ncf/README.md` and place the private ZIP in Google Drive. The archive contains ratings interactions and index mappings; do not share it publicly. Select a Colab GPU runtime before executing.

In [ ]:
import importlib.util, subprocess, sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'joblib': 'joblib'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU unavailable. Select Runtime > Change runtime type > GPU, then rerun.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Properties:', torch.cuda.get_device_properties(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from pathlib import Path
import sys

# Change BUNDLE_ZIP to the private Drive location where you uploaded the export.
BUNDLE_ZIP = Path('/content/drive/MyDrive/CineMind-NCF/cinemind_ncf_candidate_20261011.zip')
RUN_DIR = Path('/content/drive/MyDrive/CineMind-NCF/run_20261011')
BUNDLE_DIR = Path('/content/cinemind_ncf_bundle')
if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(f'Upload the safe export ZIP to Drive first: {BUNDLE_ZIP}')

import hashlib, json, zipfile
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    names = zf.namelist()
    if len(names) != len(set(names)) or 'manifest.json' not in names:
        raise ValueError('Invalid bundle member list or missing manifest')
    manifest = json.loads(zf.read('manifest.json'))
    if set(names) != set(manifest['files']) | {'manifest.json'}:
        raise ValueError('Bundle members do not match manifest allow-list')
    candidate_names = {name for name in names if name.startswith('candidate/')}
    source_names = {name for name in names if name.startswith('source/')}
    expected_source_names = {
        'source/app/__init__.py', 'source/app/ml/__init__.py',
        'source/app/ml/model_store.py', 'source/app/ml/ncf_data.py',
        'source/app/ml/ncf_evaluation.py', 'source/app/ml/ncf_model.py',
        'source/app/ml/ncf_streaming.py',
    }
    import re
    if (
        'candidate/metadata.json' not in candidate_names
        or any(name != 'candidate/metadata.json' and not re.fullmatch(r'candidate/(train|val)_\d{6}\.npz', name) for name in candidate_names)
        or source_names != expected_source_names
        or set(names) != candidate_names | expected_source_names | {'train_candidate.py', 'manifest.json'}
    ):
        raise ValueError('Bundle contains files outside the Colab export allow-list')
    for name in names:
        if name.startswith('/') or '..' in Path(name).parts or '\\' in name:
            raise ValueError(f'Unsafe bundle member path: {name}')
    helper = zf.read('train_candidate.py')
    for name, details in manifest['files'].items():
        digest = hashlib.sha256()
        size = 0
        with zf.open(name) as member:
            for chunk in iter(lambda: member.read(1024 * 1024), b''):
                digest.update(chunk)
                size += len(chunk)
        if size != details['bytes'] or digest.hexdigest() != details['sha256']:
            raise ValueError(f'Bundle integrity check failed for {name}')
helper_path = Path('/content/train_candidate.py')
helper_path.write_bytes(helper)
sys.path.insert(0, '/content')
import train_candidate as training_helpers
benchmark_candidate = training_helpers.benchmark_candidate
extract_bundle = training_helpers.extract_bundle
train_candidate = training_helpers.train_candidate

bundle_dir = extract_bundle(BUNDLE_ZIP, BUNDLE_DIR)
print('Verified bundle extracted to:', bundle_dir)
print('Run/checkpoint/model outputs:', RUN_DIR)

In [ ]:
# Short benchmark only: this does not start full training or save a model.
benchmark = benchmark_candidate(bundle_dir, max_batches=16)

## Optional full training — explicit opt-in required

Review the measured runtime and confirm Drive has room for epoch checkpoints plus the candidate artifact. Leave `RUN_FULL_TRAINING` false to stop after benchmarking. For a new run use `RESUME = False`; after an interruption, rerun with `RESUME = True` and the same bundle/settings. Do not change dataset, model or optimizer settings when resuming.

In [ ]:
RUN_FULL_TRAINING = False
RESUME = False

if not RUN_FULL_TRAINING:
    print('Full training is disabled. Set RUN_FULL_TRAINING = True only after reviewing the benchmark.')
else:
    confirmation = input('Type TRAIN CANDIDATE to start/resume the GPU training run: ')
    if confirmation != 'TRAIN CANDIDATE':
        raise RuntimeError('Confirmation did not match; no training was started.')
    report = train_candidate(bundle_dir, RUN_DIR, resume=RESUME)

In [ ]:
report_path = RUN_DIR / 'training_report.json'
if report_path.is_file():
    import json
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
else:
    print('No training report exists. Benchmarking alone does not produce evaluation results.')